# Tutorial 1.6: Framework Integrations
- [SOURCE](https://github.com/dmatrix/mlflow-genai-tutorials/blob/main/06_framework_integrations.ipynb)

![](images/7_Framework-Integrations.png)

## Working with Multiple GenAI Frameworks

MLflow supports 30+ GenAI frameworks. This notebook shows how to work with the most popular ones.

**The idea:** build agents with whichever integrated framework best fits your use case, then observe and evaluate them with MLflow.

### What You'll Learn
- Overview of MLflow's framework integrations
- Working with LangChain (chains and agents)
- Working with LlamaIndex (document indexing and RAG)
- Working with LangGraph (stateful graph workflows)
- Comparing frameworks and choosing the right one
- Best practices for each framework

### Prerequisites
- Completed previous notebooks (1.1-1.5)
- OpenAI API key or Databricks AI Gateway configured

### Estimated Time: 15-20 minutes

---
## Step 1: Framework Overview

### MLflow's 30+ [Integrations](https://mlflow.org/docs/latest/genai/tracing/integrations/)

MLflow provides automatic tracing for:

**LLM Providers:**
- OpenAI, Anthropic, Cohere, Azure OpenAI
- AWS Bedrock, Google Vertex AI
- Ollama, vLLM

**Frameworks:**
- LangChain, LangGraph, LlamaIndex, Haystack
- DSPy, AutoGen, CrewAI
- Guardrails AI, Phoenix

### Comparison Matrix

| Feature | OpenAI | LangChain | LangGraph | LlamaIndex |
|---------|--------|-----------|-----------|------------|
| **Use Case** | Direct calls | Workflows | Agents/routing | Doc Q&A |
| **Tracing** | ✅ Auto | ✅ Auto | ✅ Auto | ✅ Auto |
| **Agents** | Manual | ✅ Built-in | ✅ Built-in | ✅ Built-in |
| **RAG** | Manual | ✅ Built-in | ✅ Built-in | ✅ Built-in |
| **Stateful graphs** | ❌ | ❌ | ✅ Built-in | ❌ |

**OpenAI Direct API:**
- Simple Q&A applications
- Maximum control over prompts
- Lowest latency
- Custom implementations

**LangChain:**
- Complex multi-step and chained workflows
- Agent applications with tools
- Rapid prototyping

**LangGraph:**
- Workflows that branch based on LLM output
- Iterative refinement (loops/cycles)
- Multi-agent and Supervised orchestration
- Any workflow that benefits from explicit state management

**LlamaIndex:**
- Document-heavy applications
- Advanced indexing strategies
- Multiple data sources
- Knowledge management

---
## Step 2: Environment Setup

In [0]:
# Install additional frameworks
%pip install langchain langchain-openai

print("✅ Frameworks installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.8/571.8 kB 16.5 MB/s eta 0:00:00
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.12.2
    Not uninstalling typing-extensions at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-093af486-bb77-4bcd-b8e3-d6e4ea93f2fd
    Can't uninstall 'typing_extensions'. No files were found to uninstall.
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.6
    Not uninstalling langchain-core at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-093af486-bb77-4bcd-b8e3-d6e4ea93f2fd
    Can't uninstall 'langchain-core'. No files were found to uninstall.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
✅ Frameworks installed


In [0]:
import os
import mlflow
from openai import OpenAI
from dotenv import load_dotenv
import time

# Load environment variables
load_dotenv()

# Configure MLflow - using Databricks-managed MLflow on serverless compute
mlflow.set_tracking_uri("databricks")

# Choose your model provider:
# Option 1: "databricks" - Use Databricks Foundation Model APIs (open-source models, pay-per-token, cheaper than OpenAI)
# Option 2: "openai" - Use OpenAI directly (requires API key, more expensive)
model_provider = os.getenv("MODEL_PROVIDER", "databricks").lower()

if model_provider == "databricks":
    # Use Databricks Foundation Model APIs (Llama, GPT-OSS, Qwen, etc.)
    from databricks.sdk import WorkspaceClient
    w = WorkspaceClient()
    client = w.serving_endpoints.get_open_ai_client()
    
    # Available open-source models on databricks:
    # - databricks-gpt-oss-120b (large, high-quality)
    # - databricks-gpt-oss-20b (smaller, faster)
    # - databricks-llama-4-maverick (Llama 4)
    # - databricks-qwen35-122b-a10b (Qwen 3.5)
    model_name = "databricks-llama-4-maverick"  # Plain text output (no reasoning blocks)
    
else:
    # Use OpenAI directly
    client = OpenAI()
    model_name = "gpt-4o-mini"  # or gpt-4o, gpt-3.5-turbo, etc.
    
    if not os.getenv("OPENAI_API_KEY"):
        raise ValueError("OPENAI_API_KEY not found. Set MODEL_PROVIDER=databricks or add your OpenAI key.")

print(f"✅ Environment configured: using {model_provider.upper()} provider")
print(f"   MLflow version: {mlflow.__version__}")
print(f"   Tracking URI: {mlflow.get_tracking_uri()} (Databricks-managed)")
print(f"   Model: {model_name}")

✅ Environment configured: using DATABRICKS provider
   MLflow version: 3.8.1
   Tracking URI: databricks (Databricks-managed)
   Model: databricks-llama-4-maverick


---
## Step 3: LangChain Framework

LangChain provides abstractions for building LLM applications. Let's create a simple chain of a couple of operations.

In [0]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Enable LangChain autologging in mlflow
mlflow.langchain.autolog()

print("✅ LangChain autologging enabled")

✅ LangChain autologging enabled


In [0]:
from langchain_core.runnables import RunnableLambda
from langchain_core.messages import HumanMessage

# Simple LangChain chain
print("\n🔗 LangChain Example 1: Simple Chain\n")

# Create prompt template
prompt = ChatPromptTemplate.from_template(
    "You are a {role}. Answer: {question}"
)

# Create LLM LangChain object using the OpenAI client from Step 2
def _call_llm(prompt_value):
    text = prompt_value.to_string() if hasattr(prompt_value, "to_string") else str(prompt_value)
    response = client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": text}],
        temperature=1.0
    )
    return HumanMessage(content=response.choices[0].message.content)

llm = RunnableLambda(_call_llm)

# Create chain using LCEL (LangChain Expression Language)
chain = prompt | llm | StrOutputParser()

# Run chain (automatically traced!)
# variables role and question are passed to the prompt template
# during this invocation, the LLM will be called with the prompt template
# and the variables will be substituted in the prompt template
result = chain.invoke({
    "role": "MLflow expert",
    "question": "What makes LangChain different from using OpenAI directly?"
})

print(result)
print("\n✅ Chain execution fully traced!")
print("   - Prompt construction")
print("   - LLM call")
print("   - Output parsing")


🔗 LangChain Example 1: Simple Chain

As an MLflow expert, I'd be happy to dive into the differences between LangChain and directly using OpenAI.

LangChain is a framework that simplifies the development of applications using large language models (LLMs) like those provided by OpenAI. When you use OpenAI directly, you're essentially interacting with their APIs to leverage their models for tasks such as text generation, summarization, or question-answering. 

Here are some key aspects that differentiate LangChain from directly using OpenAI:

1. **Abstraction and Simplification**: LangChain provides a higher level of abstraction. It simplifies the process of integrating LLMs into your applications by offering a more user-friendly interface and handling some of the complexities associated with directly interacting with OpenAI's APIs.

2. **Flexibility and Interoperability**: LangChain is designed to be model-agnostic, meaning it allows you to switch between different LLMs (not just OpenAI

Trace(trace_id=tr-81106399c36321c6cf4a783565cea9d7)

### More complex chain with multiple steps
This is the typical use of LangChain's `|` operator to string together a series of operations.


In [0]:
# More complex chain with multiple steps
print("\n🔗 LangChain Example 2: Multi-Step Chain\n")


# Step 1: Generate topic
topic_prompt = ChatPromptTemplate.from_template(
    "Generate a technical topic about {domain}"
)
topic_chain = topic_prompt | llm | StrOutputParser()

# Step 2: Create outline
outline_prompt = ChatPromptTemplate.from_template(
    "Create a 3-point outline for: {topic}"
)
outline_chain = outline_prompt | llm | StrOutputParser()

# Execute pipeline in sequence in Step 2.
topic = topic_chain.invoke({"domain": "LLMOps"})
print(f"Topic: {topic}\n")

outline = outline_chain.invoke({"topic": topic})
print(f"Outline:\n{outline}")

print("\n✅ Multi-step chain traced!")
print("   Each chain creates separate spans")
print("   Full execution visible in MLflow UI")


🔗 LangChain Example 2: Multi-Step Chain

Topic: Here is a technical topic about LLMOps:

**Topic:** "Optimizing Large Language Model (LLM) Inference Pipelines using Automated Model Pruning and Knowledge Distillation Techniques in LLMOps"

**Description:** As Large Language Models (LLMs) continue to grow in size and complexity, optimizing their inference pipelines has become a critical challenge in LLMOps. Model pruning and knowledge distillation are two techniques that have shown promise in reducing the computational requirements of LLMs while maintaining their accuracy. This topic explores the application of automated model pruning and knowledge distillation techniques to optimize LLM inference pipelines in LLMOps. Specifically, it investigates the use of reinforcement learning-based pruning methods and teacher-student architectures for knowledge distillation to reduce the latency and computational costs associated with LLM inference. The topic also discusses the integration of these

[Trace(trace_id=tr-4b5abac766fa889c6777811191dd93ad), Trace(trace_id=tr-329825951ab7175c140ebae438bd4bf9)]

### LangChain Strengths

- **Abstractions**: reusable components
- **Chains**: complex multi-step workflows
- **Agents**: built-in agent patterns (which tools to call, in what order)
- **Tools**: easy tool integration

---
## Step 4: LlamaIndex Framework

LlamaIndex specializes in document indexing and retrieval-augmented generation (RAG). Let's
explore how to build one using LlamaIndex.

In [0]:
%uv pip install llama-index llama-index-llms-openai llama-index-embeddings-openai

Using Python 3.12.3 environment at: /local_disk0/.ephemeral_nfs/envs/pythonEnv-093af486-bb77-4bcd-b8e3-d6e4ea93f2fd
Resolved 68 packages in 604ms
 Downloaded pydantic-core
 Downloaded sqlalchemy
 Downloaded nltk
 Downloaded networkx
 Downloaded llama-index-core
Prepared 23 packages in 465ms
Uninstalled 3 packages in 3ms
Installed 23 packages in 105ms
 + aiosqlite==0.22.1
 + banks==2.5.1
 + colorama==0.4.6
 + dirtyjson==1.0.8
 + filetype==1.2.0
 + greenlet==3.5.6
 + griffe==2.3.0
 + griffecli==2.3.0
 + griffelib==2.3.0
 + llama-index==0.14.24
 + llama-index-core==0.14.24
 + llama-index-embeddings-openai==0.6.0
 + llama-index-instrumentation==0.6.0
 + llama-index-llms-openai==0.7.10
 + llama-index-workflows==2.24.0
 + networkx==3.6.1
 + nltk==3.10.3
 - pydantic==2.10.6
 + pydantic==2.13.5
 - pydantic-core==2.27.2
 + pydantic-core==2.46.5
 - setuptools==78.1.1
 + setuptools==84.0.0
 + sqlalchemy==2.0.54
 + tinytag==2.3.2
 + typing-inspection==0.4.4
Note: you may need to restart the kernel

Pre-installed Databricks Runtime package(s) replaced: setuptools: 78.1.1 -> 84.0.0, pydantic: 2.10.6 -> 2.13.5, pydantic_core: 2.27.2 -> 2.46.5. uv pip install may report the previous version as uninstalled, but the runtime's preinstalled packages remain on disk.


In [0]:
import os
import mlflow
from llama_index.core import VectorStoreIndex, Document, Settings
from llama_index.llms.openai import OpenAI as LlamaIndexOpenAI
from llama_index.embeddings.openai import OpenAIEmbedding

# Enable LlamaIndex autologging
mlflow.llama_index.autolog()

# Configure LlamaIndex Settings based on provider
# LlamaIndex defaults to OpenAI, so we need to explicitly configure it for Databricks
if model_provider == "databricks":
    _auth_headers = w.config.authenticate()
    _token = _auth_headers["Authorization"].replace("Bearer ", "")
    _base_url = f"{w.config.host}/serving-endpoints"
    
    Settings.llm = LlamaIndexOpenAI(
        model=model_name,
        api_key=_token,
        api_base=_base_url
    )
    Settings.embed_model = OpenAIEmbedding(
        model_name="databricks-bge-large-en",
        api_key=_token,
        api_base=_base_url
    )
    print("✅ LlamaIndex configured for Databricks Foundation Model APIs")
    print(f"   LLM: {Settings.llm.model}")
    print(f"   Embeddings: {Settings.embed_model.model_name}")
else:
    # Use default OpenAI settings (requires OPENAI_API_KEY)
    print("✅ LlamaIndex using default OpenAI configuration")

print("✅ LlamaIndex autologging enabled")

✅ LlamaIndex configured for Databricks Foundation Model APIs
   LLM: databricks-llama-4-maverick
   Embeddings: databricks-bge-large-en
✅ LlamaIndex autologging enabled


In [0]:
%uv pip install llama-index-llms-openai-like

Using Python 3.12.3 environment at: /local_disk0/.ephemeral_nfs/envs/pythonEnv-093af486-bb77-4bcd-b8e3-d6e4ea93f2fd
Resolved 67 packages in 79ms
Prepared 1 package in 11ms
Installed 1 package in 1ms
 + llama-index-llms-openai-like==0.8.0
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Override Settings.llm with OpenAILike to bypass OpenAI model name validation
# OpenAILike is designed for OpenAI-compatible endpoints with custom model names
from llama_index.llms.openai_like import OpenAILike
Settings.llm = OpenAILike(
    model=model_name,
    api_key=_token,
    api_base=_base_url,
    is_chat_model=True,
    temperature=0.1
)

# Create sample documents. Normally, you get PDFs, markdown files, HTML, or text files.
# For this example, we'll just use text for simplicity.

print("\n📚 LlamaIndex Example: Document Q&A\n")

documents = [
    Document(text="MLflow is an open source AI platform for the complete GenAI lifecycle. It provides experiment tracking, prompt registry, and agent evaluation capabilities."),
    Document(text="MLflow Tracing captures the complete execution of GenAI applications, including LLM calls, retrieval steps, and tool usage."),
    Document(text="MLflow integrates with 30+ frameworks including OpenAI, LangChain, LlamaIndex, and more."),
    Document(text="MLflow supports collaborative development with experiment sharing, prompt management and versioning."),
    Document(text="MLflow is open source and supported by Databricks. It's also OpenTelemetry-compatible, so you can monitor in production without vendor lock-in."),
]

# Create index (automatically traced)
# Note: This uses the LLM and embedding model configured in Settings above
index = VectorStoreIndex.from_documents(documents)

# Create query engine associated with the index
query_engine = index.as_query_engine()

# Query the index (automatically traced)
response = query_engine.query("What tracing capabilities does MLflow have?")

print("Query: What tracing capabilities does MLflow have?")
print(f"\nAnswer: {response}")

print("\n✅ LlamaIndex execution fully traced!")
print("   - Document indexing")
print("   - Query embedding")
print("   - Retrieval")
print("   - Response synthesis")


📚 LlamaIndex Example: Document Q&A

Query: What tracing capabilities does MLflow have?

Answer: MLflow Tracing captures the complete execution of GenAI applications, including LLM calls, retrieval steps, and tool usage.

✅ LlamaIndex execution fully traced!
   - Document indexing
   - Query embedding
   - Retrieval
   - Response synthesis


[Trace(trace_id=tr-533031b132c2438f08d0fd708b887ace), Trace(trace_id=tr-8e2f948a3a582d4342239add8574f4c5), Trace(trace_id=tr-35f2959828a234f2abb2cdc845376e8b)]

---
## Step 5: LangGraph Framework

LangGraph builds stateful, multi-node graphs where LLM output controls which path to take — ideal for branching workflows, iterative refinement, and multi-agent orchestration.

This example shows a **customer service triage bot**: an incoming message is classified, then routed to the appropriate specialist node (billing, tech support, or general inquiry).

### Customer Service Triage — Graph Flow

![Customer Service Triage Graph Flow](images/langgraph_customer_triage.svg)

Each invocation follows exactly one path: `START → classify → [handler] → END`. 

            
Every node function receives the current state, reads the fields it needs, and returns a partial dict with only the fields it updates. LangGraph merges that update back into the shared state before passing it to the next node, so the classify node can write `category` for the router to branch on, and the selected handler can write `response` for the caller, all without any node needing direct knowledge of the others.

In [0]:
# Restore typing.TypedDict if previously monkey-patched
import importlib, typing, typing_extensions
if typing.TypedDict is typing_extensions.TypedDict:
    importlib.reload(typing)

from typing import TypedDict, Literal

# Patch langchain_protocol: remove extra_items (Python 3.13+ only)
import importlib.util, pathlib, sys
_spec = importlib.util.find_spec("langchain_protocol")
if _spec:
    _proto_file = pathlib.Path(_spec.origin).parent / "protocol.py"
    _src = _proto_file.read_text()
    _src = _src.replace("\nfrom typing_extensions import TypedDict", "")
    _src = _src.replace(", extra_items=MetadataScalar", "")
    _proto_file.write_text(_src)
for _mod in list(sys.modules.keys()):
    if _mod.startswith("langchain_protocol") or _mod.startswith("langgraph"):
        del sys.modules[_mod]

from langgraph.graph import StateGraph, END
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# ── Shared state ─────────────────────────────────────────────────────────────
class CustomerServiceState(TypedDict):
    message: str
    category: str    # filled by classify node
    response: str    # filled by respective handler node

# ── Node 1: classify agent for the incoming message ────────────────────────────────────
def classify(state: CustomerServiceState) -> CustomerServiceState:
    prompt = ChatPromptTemplate.from_template(
        "You are a customer service triage agent. "
        "Classify the following message into exactly one of this category: "
        "'billing', 'tech_support', or 'customer_inquiry'. "
        "Reply with one word only.\n\nMessage: {message}"
    )
    # llm is a LangChain object
    chain = prompt | llm | StrOutputParser()
    category = chain.invoke({"message": state["message"]}).strip().lower()
    # Normalize to a known category
    if "billing" in category:
        category = "billing"
    elif "tech" in category:
        category = "tech_support"
    else:
        category = "customer_inquiry"
    return {"category": category}

# ── Node 2a: billing agent specialist ──────────────────────────────────────────────
def handle_billing(state: CustomerServiceState) -> CustomerServiceState:
    prompt = ChatPromptTemplate.from_template(
        "You are a billing specialist. Help the customer with their billing issue.\n\n"
        "Customer message: {message}"
    )
    chain = prompt | llm | StrOutputParser()
    return {"response": chain.invoke({"message": state["message"]})}

# ── Node 2b: technical support engineer agent ──────────────────────────────────────
def handle_tech_support(state: CustomerServiceState) -> CustomerServiceState:
    prompt = ChatPromptTemplate.from_template(
        "You are a technical support engineer. Help the customer resolve their technical issue.\n\n"
        "Customer message: {message}"
    )
    chain = prompt | llm | StrOutputParser()
    return {"response": chain.invoke({"message": state["message"]})}

# ── Node 2c: customer success agent ──────────────────────────────────────────
def handle_customer_inquiry(state: CustomerServiceState) -> CustomerServiceState:
    prompt = ChatPromptTemplate.from_template(
        "You are a customer success agent. Answer the customer's question helpfully.\n\n"
        "Customer message: {message}"
    )
    chain = prompt | llm | StrOutputParser()
    return {"response": chain.invoke({"message": state["message"]})}

# ── Routing function ──────────────────────────────────────────────────────────
def route(state: CustomerServiceState) -> Literal["handle_billing", "handle_tech_support", "handle_customer_inquiry"]:
    routes = {
        "billing": "handle_billing",
        "tech_support": "handle_tech_support",
        "customer_inquiry": "handle_customer_inquiry",
    }
    return routes.get(state["category"], "handle_customer_inquiry")

# ── Build and compile the graph ───────────────────────────────────────────────
# Create a StateGraph builder -- specialized for type-checking so that the state is CustomerServiceState, 
# there's no context, and both input and output are CustomerServiceState -- and pass CustomerServiceState 
# as the actual state schema

builder = StateGraph[CustomerServiceState, None, CustomerServiceState, CustomerServiceState](CustomerServiceState)
builder.add_node("classify", classify)
builder.add_node("handle_billing", handle_billing)
builder.add_node("handle_tech_support", handle_tech_support)
builder.add_node("handle_customer_inquiry", handle_customer_inquiry)

builder.set_entry_point("classify")
builder.add_conditional_edges("classify", route)
builder.add_edge("handle_billing", END)
builder.add_edge("handle_tech_support", END)
builder.add_edge("handle_customer_inquiry", END)

app = builder.compile()

# ── Run sample messages (one per route) ───────────────────────────────────────
print("🔀 LangGraph Example: Customer Service Triage\n")

test_messages = [
    "I was charged twice for my subscription this month.",
    "The app crashes whenever I try to export a report.",
    "Can you explain the difference between the Pro and Enterprise plans?",
]

for msg in test_messages:
    result = app.invoke({"message": msg})
    print(f"Message  : {result['message']}")
    print(f"Category : {result['category']}")
    print(f"Response : {result['response'][:200]}...")
    print()

print("✅ LangGraph execution fully traced!")
print("   🔍 View in MLflow UI — each invocation produces a hierarchical trace:")
print("      - Root span: full graph invocation")
print("      - 'classify' span: LLM call that picks the route")
print("      - 'handle_billing' / 'handle_tech_support' / 'handle_customer_inquiry': routed handler")

### LangGraph Strengths

- **Stateful graphs**: shared state flows between nodes via `TypedDict`
- **Conditional routing**: branch to different nodes based on state
- **Cycles**: support for loops and iterative refinement
- **Visibility**: each node appears as its own span in the MLflow trace
- **LangChain compatible**: reuses LangChain prompts, LLMs, and parsers

---
## Step 6: Best Practices

### OpenAI Best Practices

```python
# DO:
✅ Use structured prompts
✅ Implement retry logic
✅ Handle rate limits
✅ Stream responses for UX
✅ Cache results when possible

# DON'T:
❌ Hardcode prompts
❌ Ignore error responses
❌ Skip cost tracking
❌ Use synchronous calls in production
```

### LangChain Best Practices

```python
# DO:
✅ Use LCEL for chains
✅ Leverage built-in components
✅ Test chains independently
✅ Use async for better performance
✅ Enable debug mode during development

# DON'T:
❌ Over-abstract simple use cases
❌ Ignore performance overhead
❌ Skip error handling in chains
❌ Use deprecated components
```

### LangGraph Best Practices

```python
# DO:
✅ Use TypedDict for shared state — keep fields minimal
✅ Use conditional edges for branching logic
✅ Enable mlflow.langchain.autolog() to trace each node
✅ Test nodes independently before wiring the graph
✅ Use cycles sparingly — add a step counter to prevent infinite loops

# DON'T:
❌ Use LangGraph for simple linear chains (LCEL is simpler)
❌ Store large objects in state (keep it lightweight)
❌ Forget to handle the case where routing falls through
```

### LlamaIndex Best Practices

```python
# DO:
✅ Choose appropriate index type for your use case
✅ Chunk documents thoughtfully
✅ Use metadata for filtering
✅ Cache embeddings when possible
✅ Monitor retrieval quality

# DON'T:
❌ Index without preprocessing
❌ Use default settings blindly
❌ Ignore index refresh strategies
❌ Skip evaluation of retrievals
```

---
## Summary

In this notebook, you learned:

1. An overview of MLflow's 30+ framework integrations
2. Working with LangChain (chains and workflows)
3. Working with LlamaIndex (document indexing and RAG)
4. Working with LangGraph (stateful graphs with conditional routing)
5. Best practices for each framework

### Key Takeaways

- **All frameworks** are automatically traced by MLflow
- **Choose based on use case**, not hype
- **OpenAI** for simplicity and performance (used throughout the series)
- **LangChain** for complex workflows and agents
- **LangGraph** for stateful, branching, or cyclical agent workflows
- **LlamaIndex** for document-heavy applications
- **Mix frameworks** when it makes sense

### What's Next?

**Notebook 1.7: Evaluating Agents**

Learn how to evaluate your GenAI applications:
- LLM-as-Judge evaluation patterns
- MLflow built-in scorers
- Custom scorers with the `@scorer` decorator
- DeepEval integration

### Additional Resources

- [MLflow LangChain Integration](https://mlflow.org/docs/latest/llms/langchain/index.html)
- [MLflow LlamaIndex Integration](https://mlflow.org/docs/latest/llms/llama-index/index.html)
- [LangGraph Documentation](https://langchain-ai.github.io/langgraph/)
- [Framework Examples](https://github.com/mlflow/mlflow/tree/master/examples)